# Bachelor Thesis

© 2026 Yvan Richard   
University of St. Gallen, Spring Term 2026

## Reproduction of Barber et al. (2022)

---

## Foreword

In this notebook, the purpose is to reproduce Table I in Barber et al. (2022). The table depicts the summary statistics of the main data set.

## 1. Libraries & Data

I first load the relevant libraries and data.

In [17]:
# libs 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
import seaborn as sns
from datetime import datetime

# data
main = pd.read_csv("../../data/processed/CRSP_RH_TAQ_merged.csv")

# parses dates
main["date"] = pd.to_datetime(main["date"], format="%Y-%m-%d")

/var/folders/7v/_v_y1jpx0rl056gg5rkjsw4r0000gn/T/ipykernel_13091/2839584429.py:11: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  main = pd.read_csv("../../data/processed/CRSP_RH_TAQ_merged.csv")


Once the data are loaded, I build the relevant variables for computing the summary statistics.

In [18]:
# users_close_l1: lagged users_close by 1 day
main["users_close_l1"] = main.groupby("ticker")["users_close"].shift(1)

# userchg: change in users from day t-1 to day t
main["userchg"] = main["users_close"] - main["users_close_l1"]

# userratio: users_close(t) / users_close(t-1)
main["userratio"] = main["users_close"] / main["users_close_l1"]

# replace inf and -inf in userratio with NaN
main["userratio"] = main["userratio"].replace([np.inf, -np.inf], np.nan)

# make sure that I take the absolute value of the price
main["prc"] = main["prc"].abs()

# size is defined as prc * shrout
main["size"] = main["prc"] * main["shrout"] / 1e6  # in millions

# build 'daily_buys'
main["daily_buys"] = main["buy_num_trades_retail"]

# build 'daily_sells'
main["daily_sells"] = main["sell_num_trades_retail"]

# build 'net_buys'
main["net_buys"] = main["daily_buys"] - main["daily_sells"]

# build 'taq_retimb'
main['taq_retimb'] = main['net_buys'] / (main['daily_buys'] + main['daily_sells'])

# build binary variable 'has_users' indicating whether there are any users for that ticker on that day
main["has_users"] = (main["users_close"] > 0).astype(int)

I now compute Panel A

In [19]:
# summary stats Panel A
summary_stats_panel_a = main[["users_close", "users_last", "userchg", "userratio", "prc", "size", "ret",
                                "daily_buys", "daily_sells", "net_buys", "taq_retimb"]].describe().T
print(summary_stats_panel_a)

                 count         mean           std           min        25%  \
users_close  3646951.0  2213.116625  16190.180816       0.00000  42.000000   
users_last   3654694.0  2212.205310  16187.125219       0.00000  42.000000   
userchg      3617606.0     9.841856    387.446711 -548766.00000  -1.000000   
userratio    3582138.0     1.007559      0.531396       0.00000   0.997001   
prc          4431180.0    80.031313   3537.459678       0.00750   9.845000   
size         4431180.0     5.209924     29.921042       0.00003   0.074800   
ret          4431180.0     0.000364      0.043520      -1.00000  -0.009687   
daily_buys   4183850.0   193.275524   1116.786913       0.00000   8.000000   
daily_sells  4183850.0   172.821975    866.247934       0.00000   8.000000   
net_buys     4183850.0    20.453548    388.263033  -30215.00000  -7.000000   
taq_retimb   4183850.0     0.009624      0.360486      -1.00000  -0.142857   

                    50%         75%            max  
users_clos

In [21]:
# Panel B: Daily Observations (Summed Variable Averaged across Days)
# keep only observations where has_users == 1
main = main[main["has_users"] == 1]

# group by date and sum the relevant variables: [users_close, users_last, userchg, daily_buys, daily_sells, net_buys]
daily_summary = main.groupby("date").agg({
    "users_close": "sum",
    "users_last": "sum",
    "userchg": "sum",
    "daily_buys": "sum",
    "daily_sells": "sum",
    "net_buys": "sum",
    "has_users": "sum"
}).reset_index()

# summary stats Panel B
summary_stats_panel_b = daily_summary[["users_close", "users_last", "userchg", "daily_buys", "daily_sells", "net_buys", "has_users"]].describe().T
print(summary_stats_panel_b)

             count          mean           std        min        25%  \
users_close  543.0  1.486396e+07  9.748804e+06  5330135.0  8250286.5   
users_last   543.0  1.487358e+07  9.758658e+06  5331379.0  8252712.0   
userchg      543.0  6.561333e+04  1.102286e+05  -190303.0    13654.0   
daily_buys   543.0  1.301062e+06  7.448834e+05   390592.0   836898.0   
daily_sells  543.0  1.159924e+06  5.914556e+05   362930.0   793382.0   
net_buys     543.0  1.411372e+05  1.724424e+05   -69746.0    38274.5   
has_users    543.0  6.650519e+03  5.841085e+02     4420.0     6133.5   

                    50%         75%         max  
users_close  11738142.0  15097295.0  41190600.0  
users_last   11739859.0  15100933.5  41202912.0  
userchg         22873.0     53178.0    798620.0  
daily_buys     932352.0   1360427.5   3945160.0  
daily_sells    888565.0   1231154.0   3180761.0  
net_buys        65551.0    141579.5    998095.0  
has_users        6589.0      7321.5      7491.0  


In [22]:
# check most popular tickers
top_tickers = main.groupby("ticker")["users_close"].sum().sort_values(ascending=False).head(10)
print(top_tickers)

ticker
ACB     211168422.0
F       192038116.0
GE      182523030.0
AAPL    136557927.0
MSFT    134402176.0
GPRO    125694624.0
FIT     113681676.0
DIS     110533627.0
AMD      93902001.0
CRON     93409958.0
Name: users_close, dtype: float64
